<a href="https://colab.research.google.com/github/rodriguesmafalda/de-zoomcamp/blob/main/module5/module5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install pyspark

In [1]:
!pip install pyspark

Question 1

In [16]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Module5BatchProcessing") \
    .getOrCreate()


print(f"The spark version: {spark.version}")

The spark version: 3.5.5


In [22]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

--2025-03-06 21:08:15--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.209.140, 54.230.209.126, 54.230.209.200, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.209.140|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64346071 (61M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2024-10.parquet.1’

yellow_tripdata_202 100%[===================>]  61.36M   131MB/s    in 0.5s    

2025-03-06 21:08:15 (131 MB/s) - ‘yellow_tripdata_2024-10.parquet.1’ saved [64346071/64346071]



Question 2

In [25]:
# Read the dataset
file_path = "yellow_tripdata_2024-10.parquet"
df = spark.read.parquet(file_path)

# repartition and save as Parquet
output_path = "output/yellow_tripdata_partitioned"
df.repartition(4).write.mode("overwrite").parquet(output_path)


# Calculate the average size of Parquet files
import os

file_sizes = [os.path.getsize(os.path.join(output_path, f)) for f in os.listdir(output_path) if f.endswith(".parquet")]
avg_size_mb = sum(file_sizes) / len(file_sizes) / (1024 * 1024)
print(f"Average Parquet File Size: {avg_size_mb:.2f} MB")


Average Parquet File Size: 23.04 MB


Question 3

In [26]:
from pyspark.sql.functions import col, to_date

# Filter trips that started on October 15th
df_filtered = df.filter(col("tpep_pickup_datetime").substr(1, 10) == "2024-10-15")

print("Trips on October 15:", df_filtered.count())

Trips on October 15: 128893


Question 4

In [37]:
# Longest trip

df.registerTempTable('trips_data')
spark.sql("""
select MAX(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))
from trips_data
""").show()

+---------------------------------------------------------------------+
|max(timestampdiff(HOUR, tpep_pickup_datetime, tpep_dropoff_datetime))|
+---------------------------------------------------------------------+
|                                                                  162|
+---------------------------------------------------------------------+



Question 5

In [29]:
# Spark UI Port
print("Spark UI runs on port: 4040")

Spark UI runs on port: 4040


In [34]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-06 21:17:51--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 54.230.209.126, 54.230.209.140, 54.230.209.72, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|54.230.209.126|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2025-03-06 21:17:51 (229 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



Question 6

In [35]:
# Least frequent pickup location zone
zone_lookup = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)
df.createOrReplaceTempView("trips")
zone_lookup.createOrReplaceTempView("zones")

least_frequent_zone = spark.sql("""
    SELECT zones.Zone, COUNT(*) as trip_count
    FROM trips
    JOIN zones ON trips.PULocationID = zones.LocationID
    GROUP BY zones.Zone
    ORDER BY trip_count ASC
    LIMIT 1
""").collect()[0][0]

print("Least Frequent Pickup Location Zone:", least_frequent_zone)

Least Frequent Pickup Zone: Governor's Island/Ellis Island/Liberty Island
